In [ ]:
reset

In [ ]:
import os
import sys
# add path to custom functions
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path+"/py_functions")
# import custom functions
from map_plot_tools import *
from line_plot_tools import *
from colorbar_funcs import *
from data_funcs import *

import xarray as xr
xr.set_options(keep_attrs=True)
import numpy as np
import metpy.calc as mp

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo
import seaborn as sns
# settings
%config InlineBackend.figure_format = 'retina'

# top level data directory
dpath0='/Users/dervlamk/OneDrive/research'
# save figs here
opath=f'{dpath0}/nam/PaleoPaleo2025_manuscript'

**read in data**
---
Precipitation isotope data: Online Isotopes in Precipitation Calculator<br>
source: https://wateriso.utah.edu/waterisotopes/index.html

Precipitation rate data: IMERG<br>
source:

Topography: USGS ETOPO05<br>
source:

In [ ]:
filen=f'{dpath0}/topo_biases/topo_files/obs.etopo5.zsurf.nc'
etopo_full=xr.open_dataset(f'{filen}').ROSE / 1000 # convert from m to km
etopo_full.attrs['units'] = 'km'
etopo_full=longitude_flip(etopo_full)
etopo_land=etopo_full.where(etopo_full>0, np.nan) # mask bathymetry
etopo_land = etopo_land.rename({'ETOPO05_Y':'lat','ETOPO05_X':'lon'})

In [ ]:
# set lat/lon bounds
lon_min=-125
lon_max=-85
lat_min=10
lat_max=42

filen='OIPC_monthly_data.nc'
oipc=xr.open_dataset(f'{dpath0}/obs_data/{filen}').isotopes[::-1,:,:].sel(Lon=slice(lon_min,lon_max), Lat=slice(lat_min,lat_max))
oipc=oipc.rename({'Lat':'lat','Lon':'lon'})
oipc=oipc.transpose('month','lat','lon')

filen='satellite/imerg/imerg.gn.timeseries.2001-2018.nc'
ds=xr.open_dataset(f'{dpath0}/obs_data/{filen}').precipitation  * 24 # convert from mm/hr to mm/day
ds=ds.sel(lon=slice(lon_min,lon_max), lat=slice(lat_min,lat_max)).groupby('time.month').mean(dim='time') # calculate monthly climatologies
del ds.attrs["units"]
ds.attrs['Units'] = 'mm/day'
imerg=ds.transpose('month','lat','lon')

In [ ]:
# calculate seasonal dD and prec values (assumes OIPC data is already flux-weighted)

# init dicitionaries
dD = {}
prec = {}
pprec = {}

seasons=np.array(['djf','jfm','mam','jja','jas','jjas','son','ann'])

# sum annual total precip
annual_total_p = imerg.sum(dim="month")

# get seasonal means of obs data
for season in seasons:
    months = get_season(season)
    dD[season] = oipc.isel(month=months).mean(dim='month') # isotopes
    prec[season] = imerg.isel(month=months).mean(dim='month') # precip
    
    if season != 'ann':
        pprec[season] = imerg.isel(month=months).sum(dim='month') / annual_total_p
        pprec[season].attrs['Units'] = '%'
        pprec[season].attrs['newname'] = 'percent of annual total'
    else:
        pass

In [ ]:
cmap_low = cmo.speed_r
cmap_up = cmo.turbid 
ccmap = combine_cmaps(cmap_low, cmap_up, range_low=[0,.825], range_up=[.15,1], n_low=128, n_up=128)
ccmap

In [ ]:
cmap_low = cmo.speed_r
cmap_up = cmo.turbid 
ccmap = combine_cmaps_white_center(cmap_low, cmap_up, range_low=[0,.9], range_up=[0,.9], n_low=128, n_up=128, n_white=3)
#ccmap = combine_cmaps(cmap_low, cmap_up, range_low=[0,.825], range_up=[.15,1], n_low=128, n_up=128)
ccmap

In [ ]:
# Core locations
clons=[-106.5183, -111.62] 
clats=[22.5183, 27.85]
cnames=np.array(['NH22P','DSDP-480/479'])
# Model Data
olon = dD['ann'].lon; olat = dD['ann'].lat
plon = prec['ann'].lon; plat = prec['ann'].lat
# settings
lw=1
arrow_kw={'arrowstyle':'->', 'color':'k', 'linewidth':3, 'shrinkB':6}
text_kw={'fontsize':14, 'fontweight':'bold', 'ha':'center'}
text_kw1={'color':'k', 'weight':'bold', 'size':16, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':18, 'ha':'center', 'va':'center'}
titles=np.array([u'Summer$-$Winter δD$_{\mathbf{prec}}$', '%Summer Precipitation'])
letters=np.array(['A','B'])
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-120., -85., 10., 37.]
tx=np.abs((map_bnds[0]-map_bnds[1])/2) + map_bnds[0]
ty=map_bnds[3]+0.5
# isotopes cmap
icmap=ccmap #cmo.balance
ivmin=-60
ivmax=60
ilevels=np.linspace(ivmin, ivmax, 31)
inorm=mpl.colors.BoundaryNorm(ilevels, icmap.N)
# precip cmap
#pcmap,_,_,_=get_settings(field='precip', diff=False)
pcmap=cmo.rain
pvmin=10
pvmax=90
plevels=np.linspace(pvmin, pvmax, 21)
pnorm=mpl.colors.BoundaryNorm(plevels, pcmap.N)


# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(11,5), layout='constrained', subplot_kw={'projection': proj})
fig.text(.5,1,'', **text_kw)

#dDp
dDdiff = dD['jas'] - dD['jfm']
cf1=ax[0].pcolormesh(olon, olat, dDdiff, cmap=icmap, norm=inorm, transform=trans)
ax[0].scatter(x=clons, y=clats, c='k', alpha=1, edgecolor='k', s=150, transform=trans, zorder=100)
ax[0].annotate(cnames[1], xy=(clons[1],clats[1]), xytext=(clons[1]-4,clats[1]-6),
               arrowprops=arrow_kw, **text_kw)
ax[0].annotate(cnames[0], xy=(clons[0],clats[0]), xytext=(clons[0]-2.5,clats[0]-6),
               arrowprops=arrow_kw, **text_kw)

# precip
cf2=ax[1].pcolormesh(plon, plat, pprec[season]*100, cmap=pcmap, norm=pnorm, transform=trans)
ax[1].scatter(x=clons, y=clats, c='k', alpha=1, edgecolor='k', s=150, transform=trans, zorder=100)
ax[1].contour(etopo_land.lon, etopo_land.lat, etopo_land,
              levels=np.linspace(1,5,6), linewidths=.5, colors='k', transform=trans)

for i in [0,1]:
    ax[i].coastlines()
    ax[i].add_feature(cfeature.BORDERS)
    #ax[i].add_feature(cfeature.STATES, linewidth=0.5)
    ax[i].text(tx, ty, titles[i], **text_kw1)
    ax[i].text(map_bnds[0], ty+1, letters[i], **text_kw2)
    ring=LinearRing(list(zip([-113., -105, -105, -113.], [19,  19,  33,  33])))
    ax[i].add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax[i].set_extent(map_bnds, crs=trans)
    if i==0:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False
        gl.xlabel_style = {'size': 11}
        gl.ylabel_style = {'size': 11} 
    if i==1:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.left_labels=False; gl.right_labels=False
        gl.xlabel_style = {'size': 11}

cbar_ax1 = fig.add_axes([0.05, -0.025, 0.45, 0.05])
cbar1 = fig.colorbar(cf1, ticks=[-60,-40,-20,0,20,40,60], orientation='horizontal', extend='both', cax=cbar_ax1)
cbar1.set_label(u'$\mathbf{\Delta}$[‰]', weight='bold', size=12, labelpad=5, rotation=0)
cbar1.ax.tick_params(labelsize=12)
for tick in cbar1.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('bold')

cbar_ax2 = fig.add_axes([0.535, -0.025, 0.45, 0.05])
cbar2 = fig.colorbar(cf2, ticks=[0,20,40,60,80], orientation='horizontal', extend='both', cax=cbar_ax2)
cbar2.set_label('[%]', weight='bold', size=12, labelpad=5, rotation=0)
cbar2.ax.tick_params(labelsize=12)
for tick in cbar2.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('bold')

#fig.text(0,-0.175, r'For $\mathit{adjusted}$ (PaleoCalAdjust) iCESM1.2 LIG (127ka) output. Pattern correlation between differences for displayed domain: r $= -0.627$')
plt.savefig(f'{opath}/modern_climo.png', dpi=1200, bbox_inches='tight')